# Class 5/12
## Setting Up Tensor Flow and Neural Networks
### Dirks Wright


In [ ]:
### set up new environment
import venv
venv.create('.venv', with_pip=True)

In [3]:
### switch to the virtual environment
import subprocess
import sys
from pathlib import Path

venv_python = Path(".venv/Scripts/python.exe")

subprocess.check_call([venv_python, "-m", "pip", "install", "tensorflow", "ipykernel"])

0

In [4]:
### register new kernel
subprocess.check_call([
    venv_python, "-m", "ipykernel", "install",
    "--user",
    "--name=tf-env",
    "--display-name=Python (tf-env)"
])

0

In [1]:
### install tensor flow
!pip install tensorflow


[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
### install required packages
!pip install pandas numpy scikit-learn

  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ---------------------------------------- 0.0/9.8 MB ? eta -:--:--
   -- ------------------------------------- 0.5/9.8 MB 2.4 MB/s eta 0:00:04
   ----- ---------------------------------- 1.3/9.8 MB 3.2 MB/s eta 0:00:03
   -------- ------------------------------- 2.1/9.8 MB 3.6 MB/s eta 0:00:03
   ----------- ---------------------------- 2.9/9.8 MB 3.6 MB/s eta 0:00:02
   ---------------- ----------------------- 3.9/9.8 MB 3.9 MB/s eta 0:00:02
   ---------------------- ----------------- 5.5/9.8 MB 4.5 MB/s eta 0:00:01
   ----------------------- ---------------- 5.8/9.8 MB 4.1 MB/s eta 0:00:01
   ------------------------ --------------- 6.0/9.8 MB 4.0 MB/s eta 0:00:01
   ------------------------- -------------- 6.3/9.8 MB 3.6 MB/s eta 0:00:01
   --------------------------------- ------ 8.1/9.8 MB 3.9 MB/s eta 0:00:01
   ------------------------------------


[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score, roc_curve

In [3]:
! pip install palmerpenguins
from palmerpenguins import load_penguins
penguins = load_penguins()
penguins.head()


[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,year
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,male,2007
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,female,2007
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,female,2007
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN,2007
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,female,2007


In [4]:
### drop nan rows
penguins = penguins.dropna()
penguins.shape

(333, 8)

In [5]:
### shuffle the data
penguins = penguins.sample(frac = 1, random_state = 42).reset_index(drop=True)

In [6]:
penguins_x = pd.concat(
    [
        penguins[['body_mass_g', 'bill_length_mm', 'bill_depth_mm', 'flipper_length_mm']],
        pd.get_dummies(penguins['sex'], dtype=int)
    ],
    axis=1
).astype(float)
penguins_x

,body_mass_g,bill_length_mm,bill_depth_mm,flipper_length_mm,female,male
0,3250.0,39.5,16.7,178.0,1.0,0.0
1,3675.0,50.9,17.9,196.0,1.0,0.0
2,4000.0,42.1,19.1,195.0,0.0,1.0
3,4850.0,46.6,14.2,210.0,1.0,0.0
4,4050.0,41.1,18.2,192.0,0.0,1.0
...,...,...,...,...,...,...
328,4750.0,49.6,15.0,216.0,0.0,1.0
329,3900.0,37.2,19.4,184.0,0.0,1.0
330,3200.0,39.7,17.7,193.0,1.0,0.0
331,3950.0,45.2,17.8,198.0,1.0,0.0


In [7]:
penguins_y = penguins['species'].astype('category').cat.codes.to_numpy()

X_train, X_val, y_train, y_val = train_test_split(
    penguins_x,
    penguins_y,
    test_size=0.2,
    random_state=42,
    stratify=penguins_y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

X_train_scaled.shape, X_val_scaled.shape

((266, 6), (67, 6))

In [8]:
pd.DataFrame({
    'train_count': pd.Series(y_train).value_counts().sort_index(),
    'validation_count': pd.Series(y_val).value_counts().sort_index()
})

,train_count,validation_count
0,117,29
1,54,14
2,95,24


In [9]:
#construct the model
inputs = keras.Input(shape=(X_train_scaled.shape[1],))
x = layers.Dense(7, activation = 'relu')(inputs)
x = layers.Dense(5, activation = 'relu')(x)
x = layers.Dense(3, activation = 'relu')(x)
outputs = layers.Dense(3, activation='softmax')(x)
model = keras.Model(inputs=inputs, outputs=outputs, name="penguin_model")

In [10]:
keras.utils.plot_model(model, show_shapes = True)

You must install pydot (`pip install pydot`) for `plot_model` to work.


In [11]:
model.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    optimizer=keras.optimizers.RMSprop(),
    metrics=["accuracy"],
)

history = model.fit(
    X_train_scaled,
    y_train,
    batch_size=64,
    epochs=100,
    validation_data=(X_val_scaled, y_val)
)

scores = model.evaluate(X_val_scaled, y_val, verbose=2)

Epoch 1/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 40ms/step - accuracy: 0.5639 - loss: 1.0931 - val_accuracy: 0.5821 - val_loss: 1.0814
Epoch 2/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.5752 - loss: 1.0820 - val_accuracy: 0.5821 - val_loss: 1.0716
Epoch 3/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.5789 - loss: 1.0732 - val_accuracy: 0.5821 - val_loss: 1.0630
Epoch 4/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.5789 - loss: 1.0652 - val_accuracy: 0.5821 - val_loss: 1.0553
Epoch 5/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.5789 - loss: 1.0577 - val_accuracy: 0.5821 - val_loss: 1.0469
Epoch 6/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.5977 - loss: 1.0490 - val_accuracy: 0.5821 - val_loss: 1.0373
Epoch 7/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.6015 - loss: 1.0394 - val_accuracy: 0.5821 - val_loss: 1.0281
Epoch 8/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.6053 - loss: 1.0300 - val_accuracy: 0.5821 - val_loss:

In [12]:
inputs_logit_true = keras.Input(shape=(X_train_scaled.shape[1],))
x_logit_true = layers.Dense(7, activation='relu')(inputs_logit_true)
x_logit_true = layers.Dense(5, activation='relu')(x_logit_true)
x_logit_true = layers.Dense(3, activation='relu')(x_logit_true)
outputs_logit_true = layers.Dense(3, activation='softmax')(x_logit_true)
model_logit_true = keras.Model(inputs=inputs_logit_true, outputs=outputs_logit_true, name="penguin_model_scaled")

model_logit_true.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    optimizer=keras.optimizers.RMSprop(),
    metrics=["accuracy"],
)

history_logit_true = model_logit_true.fit(
    X_train_scaled,
    y_train,
    batch_size=64,
    epochs=100,
    validation_data=(X_val_scaled, y_val)
)

scores = model_logit_true.evaluate(X_val_scaled, y_val, verbose=2)

Epoch 1/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 37ms/step - accuracy: 0.2519 - loss: 1.0734 - val_accuracy: 0.2090 - val_loss: 1.0764
Epoch 2/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.2707 - loss: 1.0481 - val_accuracy: 0.3134 - val_loss: 1.0561
Epoch 3/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.3571 - loss: 1.0278 - val_accuracy: 0.4030 - val_loss: 1.0408
Epoch 4/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.4248 - loss: 1.0107 - val_accuracy: 0.4776 - val_loss: 1.0258
Epoch 5/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.5338 - loss: 0.9950 - val_accuracy: 0.4925 - val_loss: 1.0122
Epoch 6/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.5865 - loss: 0.9810 - val_accuracy: 0.5522 - val_loss: 0.9987
Epoch 7/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.6203 - loss: 0.9668 - val_accuracy: 0.5821 - val_loss: 0.9850
Epoch 8/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.6692 - loss: 0.9518 - val_accuracy: 0.6567 - val_loss:

In [13]:
model_logit_true.predict(X_val_scaled)

3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step


array([[4.94092660e-07, 7.48484745e-04, 9.99251068e-01],
       [4.42439467e-01, 3.55196834e-01, 2.02363729e-01],
       [7.65357554e-01, 2.08587408e-01, 2.60550659e-02],
       [2.31833184e-07, 5.04514668e-04, 9.99495268e-01],
       [2.50685996e-08, 1.71554027e-04, 9.99828458e-01],
       [1.75563613e-07, 4.39495430e-04, 9.99560416e-01],
       [7.86062837e-01, 1.93076372e-01, 2.08608638e-02],
       [8.79761636e-01, 1.14760429e-01, 5.47801610e-03],
       [8.77149880e-01, 1.17079020e-01, 5.77110238e-03],
       [5.39321604e-07, 7.78975722e-04, 9.99220371e-01],
       [3.75187069e-01, 3.62607926e-01, 2.62205034e-01],
       [8.99898469e-01, 9.65908617e-02, 3.51062510e-03],
       [9.20743763e-01, 7.72655755e-02, 1.99068571e-03],
       [6.75447464e-01, 2.65382260e-01, 5.91703318e-02],
       [6.09031856e-01, 2.99545735e-01, 9.14224312e-02],
       [9.23420727e-01, 7.47480243e-02, 1.83123059e-03],
       [6.02571666e-01, 3.02456111e-01, 9.49721932e-02],
       [8.69527817e-01, 1.23794

In [14]:
### logit false
inputs_logit_false = keras.Input(shape=(X_train_scaled.shape[1],))
x_logit_false = layers.Dense(7, activation='relu')(inputs_logit_false)
x_logit_false = layers.Dense(5, activation='relu')(x_logit_false)
x_logit_false = layers.Dense(3, activation='relu')(x_logit_false)
outputs_logit_false = layers.Dense(3, activation='softmax')(x_logit_false)
model_logit_false = keras.Model(inputs=inputs_logit_false, outputs=outputs_logit_false, name="penguin_model_scaled")

model_logit_false.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    optimizer=keras.optimizers.RMSprop(),
    metrics=["accuracy"],
)

history_logit_false = model_logit_false.fit(
    X_train_scaled,
    y_train,
    batch_size=64,
    epochs=100,
    validation_data=(X_val_scaled, y_val)
)

scores = model_logit_false.evaluate(X_val_scaled, y_val, verbose=2)

Epoch 1/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 40ms/step - accuracy: 0.1692 - loss: 1.1998 - val_accuracy: 0.1791 - val_loss: 1.1774
Epoch 2/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.1805 - loss: 1.1644 - val_accuracy: 0.1791 - val_loss: 1.1524
Epoch 3/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.1805 - loss: 1.1418 - val_accuracy: 0.1791 - val_loss: 1.1321
Epoch 4/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.1729 - loss: 1.1232 - val_accuracy: 0.1791 - val_loss: 1.1162
Epoch 5/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.1617 - loss: 1.1084 - val_accuracy: 0.1642 - val_loss: 1.1002
Epoch 6/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.1429 - loss: 1.0935 - val_accuracy: 0.1791 - val_loss: 1.0875
Epoch 7/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.1391 - loss: 1.0821 - val_accuracy: 0.1791 - val_loss: 1.0746
Epoch 8/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.1617 - loss: 1.0704 - val_accuracy: 0.2090 - val_loss:

In [15]:
model_logit_false.predict(X_val_scaled)

3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step


array([[2.24977164e-04, 2.95091216e-02, 9.70265985e-01],
       [9.19167697e-02, 7.39591897e-01, 1.68491304e-01],
       [5.89278519e-01, 4.03122246e-01, 7.59924063e-03],
       [7.45050420e-05, 2.01248080e-02, 9.79800701e-01],
       [6.48830837e-06, 8.58683325e-03, 9.91406679e-01],
       [5.85901216e-05, 1.85121354e-02, 9.81429279e-01],
       [7.34783828e-01, 2.63424069e-01, 1.79207453e-03],
       [8.40479255e-01, 1.58723831e-01, 7.96976208e-04],
       [8.95970285e-01, 1.02495313e-01, 1.53444335e-03],
       [5.50839904e-05, 1.81190372e-02, 9.81825829e-01],
       [3.00312757e-01, 6.52130008e-01, 4.75572422e-02],
       [9.75848675e-01, 2.36458052e-02, 5.05421020e-04],
       [9.68969703e-01, 3.08617484e-02, 1.68518949e-04],
       [6.52961791e-01, 3.44072431e-01, 2.96575110e-03],
       [3.79219443e-01, 6.01239443e-01, 1.95411220e-02],
       [9.12133217e-01, 8.77346843e-02, 1.32135407e-04],
       [2.81895071e-01, 6.96298361e-01, 2.18065437e-02],
       [8.52831662e-01, 1.46759

In [16]:
penguins['species']

0         Adelie
1      Chinstrap
2         Adelie
3         Gentoo
4         Adelie
         ...    
328       Gentoo
329       Adelie
330       Adelie
331    Chinstrap
332       Adelie
Name: species, Length: 333, dtype: str

In [17]:
### training data
penguins_y

array([0, 1, 0, 2, 0, 1, 1, 2, 2, 2, 0, 0, 1, 0, 1, 0, 0, 2, 0, 1, 0, 0,
       1, 2, 0, 0, 2, 1, 2, 1, 2, 1, 0, 0, 1, 1, 2, 2, 0, 0, 0, 0, 2, 2,
       0, 0, 1, 0, 0, 1, 0, 2, 2, 0, 0, 2, 0, 0, 2, 2, 1, 1, 1, 0, 0, 1,
       0, 2, 0, 1, 0, 0, 2, 1, 2, 2, 0, 0, 0, 2, 0, 0, 2, 0, 1, 2, 0, 1,
       2, 2, 2, 1, 0, 0, 0, 0, 0, 2, 0, 0, 0, 1, 1, 0, 2, 0, 2, 2, 0, 2,
       0, 1, 0, 2, 2, 2, 0, 2, 0, 2, 0, 2, 1, 0, 0, 1, 0, 0, 0, 2, 0, 0,
       2, 0, 0, 0, 2, 0, 1, 0, 0, 2, 0, 1, 2, 1, 2, 1, 2, 2, 2, 2, 0, 0,
       2, 2, 2, 0, 2, 2, 0, 1, 1, 1, 2, 2, 2, 2, 2, 0, 0, 2, 1, 0, 1, 1,
       0, 0, 0, 0, 1, 0, 2, 1, 0, 2, 2, 0, 1, 0, 1, 0, 2, 0, 2, 0, 0, 0,
       2, 0, 2, 0, 1, 0, 0, 2, 2, 2, 0, 0, 0, 2, 2, 0, 0, 1, 0, 2, 0, 1,
       1, 1, 0, 2, 1, 2, 2, 0, 2, 0, 0, 2, 0, 2, 0, 2, 1, 0, 1, 2, 1, 0,
       2, 2, 2, 0, 0, 0, 2, 2, 2, 1, 2, 0, 0, 2, 0, 0, 0, 0, 0, 2, 0, 1,
       1, 2, 1, 2, 2, 2, 1, 2, 1, 1, 1, 2, 2, 0, 2, 2, 2, 0, 0, 0, 0, 0,
       2, 0, 2, 0, 0, 2, 2, 0, 0, 1, 2, 1, 0, 1, 2,